# Pre-Refusal Signatures: Qwen2.5 Colab Runner

This notebook runs the real hidden-state probing pipeline on **Qwen/Qwen2.5-1.5B-Instruct** and packages the generated artifacts into a `.zip` file.

**Expected runtime on a Colab T4:** usually a few minutes for the 80-prompt dataset, because the pipeline performs forward passes only and does not generate assistant responses.

**Output:** `pre_refusal_qwen_results.zip`, containing `figures/`, `reports/`, selected `outputs/`, config files, and run metadata.

## 1. Runtime Check

In Colab, set **Runtime -> Change runtime type -> T4 GPU** before running the notebook.

In [ ]:
import os, sys, platform, subprocess, json, textwrap

print('Python:', sys.version)
print('Platform:', platform.platform())

try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('CUDA capability:', torch.cuda.get_device_capability(0))
except Exception as exc:
    print('Torch import failed:', repr(exc))

## 2. Clone Repository

Before the GitHub repo exists, you can upload this notebook manually. After the repo exists, set `REPO_URL` to the real GitHub URL and this cell will clone it.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USER/pre-refusal-signatures.git'  # TODO: replace after GitHub repo is created
PROJECT_DIR = Path('/content/pre-refusal-signatures')

if not PROJECT_DIR.exists():
    if 'YOUR_USER' in REPO_URL:
        raise ValueError('Set REPO_URL to your actual GitHub repository URL before running this cell.')
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print('Using existing project directory:', PROJECT_DIR)

%cd /content/pre-refusal-signatures
!git status --short || true

## 3. Install Dependencies

Colab usually already has PyTorch. This cell installs the research pipeline dependencies without forcing a Torch reinstall.

In [ ]:
!python -m pip install -q --upgrade transformers accelerate scikit-learn numpy pandas matplotlib seaborn pyyaml tqdm pytest

## 4. Configure Run

Default settings should work on a T4. If you hit CUDA out-of-memory, set `MAX_PROMPTS = 40` and/or `MAX_LENGTH = 256`.

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
DEVICE = 'cuda'
MAX_PROMPTS = None  # Use None for all 80 prompts. Use 40 for a faster fallback.
MAX_LENGTH = 512    # Use 256 if T4 memory becomes tight.

import yaml
from pathlib import Path

config_path = Path('configs/default.yaml')
config = yaml.safe_load(config_path.read_text())
config['model_name'] = MODEL_NAME
config['device'] = DEVICE
config['max_length'] = MAX_LENGTH
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())

## 5. Validate Dataset

This confirms the curated dataset is balanced and schema-valid before model extraction.

In [ ]:
!python scripts/00_validate_dataset.py --data data/prompts.jsonl --min-per-label 40

## 6. Extract Qwen Hidden States

This is the main compute step. It runs a forward pass only, extracts every layer's hidden state, and stores pooled vectors in `outputs/hidden_states.npz`.

In [ ]:
cmd = ['python', 'scripts/01_extract_hidden_states.py', '--config', 'configs/default.yaml', '--device', DEVICE]
if MAX_PROMPTS is not None:
    cmd += ['--max-prompts', str(MAX_PROMPTS)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 7. Train Layer-Wise Linear Probes

This trains one logistic-regression probe per layer with stratified cross-validation.

In [ ]:
!python scripts/02_train_layer_probes.py --states outputs/hidden_states.npz

## 8. Generate Figures and Error Analysis

In [ ]:
!python scripts/03_make_figures.py --states outputs/hidden_states.npz --metrics reports/layer_probe_metrics.csv

## 9. Inspect Results

In [ ]:
import pandas as pd
from IPython.display import display, Image

metrics = pd.read_csv('reports/layer_probe_metrics.csv')
best = metrics.sort_values(['f1', 'accuracy'], ascending=False).iloc[0]
print('Best layer:', int(best.layer))
print(best[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']])
display(metrics.tail())

display(Image(filename='figures/layer_accuracy_curve.png'))
display(Image(filename='figures/best_layer_pca.png'))
display(Image(filename='figures/confusion_matrix.png'))

## 10. Run Early Guard Demo

In [ ]:
!python scripts/04_run_guard_demo.py --prompt "Explain photosynthesis simply." --device cuda

## 11. Run Tests

In [ ]:
!pytest -q

## 12. Save Run Metadata

In [ ]:
import datetime, json, os, subprocess, platform
from pathlib import Path

metadata = {
    'timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'model_name': MODEL_NAME,
    'device_requested': DEVICE,
    'max_prompts': MAX_PROMPTS,
    'max_length': MAX_LENGTH,
    'python': sys.version,
    'platform': platform.platform(),
}
try:
    import torch
    metadata['torch'] = torch.__version__
    metadata['cuda_available'] = torch.cuda.is_available()
    metadata['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception as exc:
    metadata['torch_error'] = repr(exc)

try:
    metadata['git_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
except Exception:
    metadata['git_commit'] = None

Path('reports/run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(json.dumps(metadata, indent=2))

## 13. Package Results as ZIP

Download this zip and send/extract it locally. It contains the files needed to update the repository results.

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path('/content/pre_refusal_qwen_results.zip')
include_dirs = ['figures', 'reports', 'outputs', 'configs']
include_files = ['README.md', 'data/prompts.jsonl']

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in include_dirs:
        for path in Path(folder).rglob('*'):
            if path.is_file():
                zf.write(path, arcname=str(path))
    for file_name in include_files:
        path = Path(file_name)
        if path.exists():
            zf.write(path, arcname=str(path))

print('Created:', zip_path)
print('Size MB:', round(zip_path.stat().st_size / 1_000_000, 2))

In [ ]:
from google.colab import files
files.download('/content/pre_refusal_qwen_results.zip')

## If T4 Runs Out of Memory

Use these fallback settings in section 4 and rerun from section 6 onward:

```python
MAX_PROMPTS = 40
MAX_LENGTH = 256
```

The full result is preferable, but a 40-prompt real-model run is still more meaningful than a synthetic-only repository.